# 001 · JSON

> **Standard Library › Data Formats** · Python 3.14

The `json` module converts between JSON text and plain Python objects (`dict`, `list`, `str`, `int`, `float`, `bool`, `None`). It is the Python counterpart of `System.Text.Json` — but it maps JSON onto *built-in* types rather than onto your own classes.

**In this notebook you will learn:**

- `dumps`/`loads` (strings) vs `dump`/`load` (files)
- how JSON types map to Python types, and the formatting options (`indent`, `sort_keys`, `ensure_ascii`)
- how to serialise custom types with `default=` or a `JSONEncoder` subclass
- how to decode into dataclasses with `object_hook`
- the gotchas: keys are always strings, `datetime` is not serialisable, `NaN` produces invalid JSON

## .NET mental model

| .NET (`System.Text.Json`) | Python (`json`) |
|---|---|
| `JsonSerializer.Serialize(obj)` | `json.dumps(obj)` → `str` |
| `JsonSerializer.Deserialize<T>(text)` | `json.loads(text)` → `dict`/`list`/... (**no** `T`) |
| `JsonSerializer.SerializeAsync(stream, obj)` | `json.dump(obj, file)` |
| `JsonSerializer.Deserialize<T>(stream)` | `json.load(file)` |
| `JsonDocument` / `JsonNode` | the `dict`/`list` you get back *is* the DOM |
| `JsonSerializerOptions { WriteIndented = true }` | `indent=2` |
| `JsonConverter<T>` | `default=` hook / `JSONEncoder` subclass, `object_hook=` |
| `JsonException` | `json.JSONDecodeError` (a subclass of `ValueError`) |

The big mental shift: `json.loads` always returns **untyped** built-in containers — it's closer to `JsonNode.Parse` than to `Deserialize<T>`. Mapping into your own classes is something you do yourself (or with a third-party library such as pydantic).

## Strings: `dumps` and `loads`

The trailing **s** means *string*. `dumps` = "dump to string", `loads` = "load from string".

```csharp
// C#
string json = JsonSerializer.Serialize(new { Name = "widget", Price = 9.99, Tags = new[] { "a", "b" } });
JsonNode? node = JsonNode.Parse(json);
Console.WriteLine(node!["Name"]);
```

In [ ]:
import json

product = {"name": "widget", "price": 9.99, "tags": ["a", "b"], "in_stock": True, "discount": None}

text = json.dumps(product)
print(type(text).__name__, text)

back = json.loads(text)
print(type(back).__name__, back["name"], back["tags"])
print(back == product)    # True — a round trip of plain data is lossless

## Type mapping

| JSON | Python (decoding) | Python types that encode to it |
|---|---|---|
| object `{}` | `dict` | `dict` |
| array `[]` | `list` | `list`, **`tuple`** |
| string | `str` | `str` |
| number (int) | `int` (arbitrary size) | `int`, int-derived `Enum`s |
| number (real) | `float` | `float`, float-derived `Enum`s |
| `true` / `false` | `True` / `False` | `True` / `False` |
| `null` | `None` | `None` |

Note that tuples come back as lists — JSON has only one sequence type.

In [ ]:
data = {"point": (1, 2), "big": 2**70, "ok": True, "nothing": None}
text = json.dumps(data)
print(text)                      # true / null are JSON spellings
print(json.loads(text))          # the tuple came back as a list

## Formatting options

| Option | Effect | `JsonSerializerOptions` analogue |
|---|---|---|
| `indent=2` | pretty-print | `WriteIndented = true` |
| `sort_keys=True` | stable key order (great for diffs and hashing) | — |
| `ensure_ascii=False` | keep non-ASCII characters instead of `\uXXXX` escapes | `Encoder = JavaScriptEncoder.UnsafeRelaxedJsonEscaping` |
| `separators=(",", ":")` | most compact output | default compact output |

In [ ]:
order = {"id": 42, "customer": "Zoë", "items": ["café", "crème"]}

print(json.dumps(order))                          # default: ASCII-only escapes
print(json.dumps(order, ensure_ascii=False))      # readable UTF-8
print(json.dumps(order, separators=(",", ":")))   # compact
print(json.dumps(order, indent=2, sort_keys=True, ensure_ascii=False))

## Files: `dump` and `load`

Without the **s**, the functions work on file objects. Open text files with an explicit `encoding="utf-8"` *(see Standard Library › Files & Paths › 002 · Reading and Writing Files)*.

```csharp
// C#
await using var fs = File.Create(path);
await JsonSerializer.SerializeAsync(fs, settings, new JsonSerializerOptions { WriteIndented = true });
```

In [ ]:
import tempfile
from pathlib import Path

settings = {"theme": "dark", "font_size": 14, "recent": ["a.txt", "b.txt"]}

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "settings.json"

    with path.open("w", encoding="utf-8") as f:
        json.dump(settings, f, indent=2)

    print(path.read_text(encoding="utf-8"))

    with path.open(encoding="utf-8") as f:
        loaded = json.load(f)

print(loaded == settings)

**Tip:** for small files, `json.loads(path.read_text(encoding="utf-8"))` and `path.write_text(json.dumps(obj), encoding="utf-8")` are perfectly fine one-liners.

## ⚠️ Gotcha: JSON keys are always strings

JSON object keys must be strings. Python happily serialises `int`, `float`, `bool` and `None` keys by converting them to strings — but they **do not** come back as their original type.

In [ ]:
stock_by_id = {1: "apple", 2: "banana"}

text = json.dumps(stock_by_id)
print(text)

back = json.loads(text)
print(back)
print(back == stock_by_id)        # False!
print(1 in back, "1" in back)     # False True

Convert keys back yourself when you need them, and note that tuple keys are rejected outright:

In [ ]:
fixed = {int(k): v for k, v in json.loads(text).items()}
print(fixed == stock_by_id)       # True

try:
    json.dumps({(0, 0): "origin"})
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## ⚠️ Gotcha: `datetime` (and most of your own types) are not serialisable

`System.Text.Json` knows `DateTime`, `Guid`, `decimal` and any POCO. Python's `json` knows only the built-ins in the table above — everything else raises `TypeError`.

In [ ]:
from datetime import datetime, UTC

event = {"name": "deploy", "at": datetime(2025, 3, 1, 12, 30, tzinfo=UTC)}

try:
    json.dumps(event)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Custom encoding with `default=`

`default` is a function that `json` calls for any object it does not know. Return something serialisable — or raise `TypeError` to keep the normal error. This is the lightweight equivalent of registering a `JsonConverter`.

In [ ]:
from dataclasses import dataclass, asdict
from decimal import Decimal
from uuid import UUID

@dataclass
class Order:
    id: UUID
    total: Decimal
    placed: datetime

def to_json(obj):
    match obj:
        case datetime():
            return obj.isoformat()
        case Decimal() | UUID():
            return str(obj)
        case Order():
            return asdict(obj)          # dict of the fields; values go through default again
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

order = Order(UUID("12345678-1234-5678-1234-567812345678"), Decimal("19.99"), datetime(2025, 3, 1, 12, 30, tzinfo=UTC))
print(json.dumps(order, default=to_json, indent=2))

## Custom encoding with a `JSONEncoder` subclass

If you need the same rules in many places, subclass `json.JSONEncoder`, override `default`, and pass it via `cls=`. Call `super().default(obj)` for the fallback — it raises the standard `TypeError`.

In [ ]:
class AppEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.isoformat()
        if isinstance(obj, (Decimal, UUID)):
            return str(obj)
        if isinstance(obj, set):
            return sorted(obj)
        return super().default(obj)

print(json.dumps({"tags": {"b", "a"}, "price": Decimal("1.50")}, cls=AppEncoder))

try:
    json.dumps({"x": object()}, cls=AppEncoder)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Decoding into objects with `object_hook`

There is no `Deserialize<T>`. Instead, `object_hook` is called with **every** decoded JSON object (innermost first), and whatever it returns replaces that `dict`. A common pattern is to look at the keys to decide what to build.

```csharp
// C#
var reading = JsonSerializer.Deserialize<SensorReading>(json);
```

In [ ]:
@dataclass
class SensorReading:
    sensor: str
    value: float
    taken: datetime

def as_reading(d: dict):
    if d.keys() == {"sensor", "value", "taken"}:
        return SensorReading(d["sensor"], d["value"], datetime.fromisoformat(d["taken"]))
    return d   # leave other objects as plain dicts

payload = '{"batch": 7, "readings": [{"sensor": "t1", "value": 21.5, "taken": "2025-03-01T08:00:00+00:00"}]}'
result = json.loads(payload, object_hook=as_reading)
print(result["batch"])
print(result["readings"][0])

**Tip:** for a single known shape, it is often simpler to `loads` into a dict and then call `SensorReading(**d)` yourself. For validation-heavy APIs, the third-party `pydantic` library gives you `Deserialize<T>`-style typed parsing.

Other hooks on `loads`/`load`: `parse_float=Decimal` (exact money values), `parse_int=...`, and `object_pairs_hook=` (receives a list of key/value pairs — useful for detecting duplicate keys).

In [ ]:
print(json.loads('{"price": 0.1}', parse_float=Decimal))

## Errors: `JSONDecodeError`

Malformed input raises `json.JSONDecodeError` — a subclass of `ValueError` — with the position of the problem in `.msg`, `.lineno` and `.colno`. Note that Python's parser is strict: no trailing commas, no comments, no single quotes (same as the `System.Text.Json` defaults).

In [ ]:
for bad in ['{"a": 1,}', "{'a': 1}", '{"a": 1} // note']:
    try:
        json.loads(bad)
    except json.JSONDecodeError as e:
        print(f"{type(e).__name__}: {e}  -> line {e.lineno}, col {e.colno}")

print(issubclass(json.JSONDecodeError, ValueError))

## ⚠️ Gotcha: `NaN` and `Infinity` produce invalid JSON

By default `json.dumps` writes `float("nan")` as the bare token `NaN` — which is **not** valid JSON, and other parsers (including `System.Text.Json` by default) will reject it. Use `allow_nan=False` to fail fast instead.

In [ ]:
reading = {"temp": float("nan"), "max": float("inf")}
print(json.dumps(reading))           # NaN / Infinity — not standard JSON!

try:
    json.dumps(reading, allow_nan=False)
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

## The `json` command line tool

The module doubles as a pretty-printer/validator from the shell (Python 3.14 also adds colour output and a `python -m json` shortcut):

```bash
echo '{"b": 1, "a": [1, 2]}' | python -m json.tool --sort-keys
python -m json.tool data.json          # validate + pretty-print a file
```

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Object → JSON string | `JsonSerializer.Serialize(obj)` | `json.dumps(obj)` |
| JSON string → data | `JsonNode.Parse(s)` / `Deserialize<T>(s)` | `json.loads(s)` (dict/list only) |
| Write to a file/stream | `SerializeAsync(stream, obj)` | `json.dump(obj, f)` |
| Read from a file/stream | `DeserializeAsync<T>(stream)` | `json.load(f)` |
| Pretty print | `WriteIndented = true` | `indent=2` |
| Keep non-ASCII | `UnsafeRelaxedJsonEscaping` | `ensure_ascii=False` |
| Custom type → JSON | `JsonConverter<T>.Write` | `default=fn` / `JSONEncoder` subclass |
| JSON → custom type | `JsonConverter<T>.Read` / `Deserialize<T>` | `object_hook=fn` |
| Parse error | `JsonException` | `json.JSONDecodeError` (`ValueError`) |

## Exercises

1. Serialise `{"when": date(2025, 1, 31), "ids": {3, 1, 2}}` to JSON with sorted keys, writing dates as ISO strings and sets as sorted lists.
2. Given `counts = {1: 10, 2: 20}`, round-trip it through JSON so that the result `== counts`.
3. Write an `object_hook` that turns any JSON object with exactly the keys `"x"` and `"y"` into a `Point` dataclass, and use it to decode `'{"path": [{"x": 0, "y": 0}, {"x": 3, "y": 4}]}'`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import json
from dataclasses import dataclass
from datetime import date

# 1 — a default= hook handles both unknown types
def encode(obj):
    if isinstance(obj, date):
        return obj.isoformat()
    if isinstance(obj, set):
        return sorted(obj)
    raise TypeError(f"not serialisable: {type(obj).__name__}")

print(json.dumps({"when": date(2025, 1, 31), "ids": {3, 1, 2}}, default=encode, sort_keys=True))
# {"ids": [1, 2, 3], "when": "2025-01-31"}

# 2 — keys become strings, so convert them back
counts = {1: 10, 2: 20}
restored = {int(k): v for k, v in json.loads(json.dumps(counts)).items()}
print(restored == counts)   # True

# 3 — object_hook sees every object, innermost first
@dataclass
class Point:
    x: int
    y: int

def hook(d):
    return Point(**d) if d.keys() == {"x", "y"} else d

print(json.loads('{"path": [{"x": 0, "y": 0}, {"x": 3, "y": 4}]}', object_hook=hook))
# {'path': [Point(x=0, y=0), Point(x=3, y=4)]}
```

</details>